# Tutorial 2: Spectral Attention

This notebook demonstrates the **multiscale spectral attention** mechanism in SMGP,
which replaces the standard O(N²) transformer self-attention with a graph spectral
approach achieving O(N log N) complexity.

Topics covered:

1. Build a graph from token embeddings.
2. Run spectral attention forward pass.
3. Examine hierarchical coarsening (graph pyramid).
4. Compare attention weights across heads.

**References:**
- Vaswani, A., et al. (2017). "Attention Is All You Need." *NeurIPS*.
- Shuman, D.I., et al. (2013). "Signal Processing on Graphs." *IEEE SPM*.
- Hammond, D.K., et al. (2011). "Wavelets on Graphs." *ACHA*.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
import smgp
print(f"SMGP version: {smgp.__version__}")

SMGP version: 0.1.0


## 1. Building a Graph from Token Embeddings

The `SpectralAttention` class constructs a graph from input token embeddings.
Each token becomes a node, and edges are created based on k-nearest-neighbour
similarity in the HD vector space (or from a pre-computed attention matrix).

Let us create synthetic token embeddings and build the context graph.

In [2]:
from smgp.core.graph import SpectralMemoryGraph
from smgp.attention.spectral_attn import SpectralAttention

# Create a base knowledge graph
base_graph = SpectralMemoryGraph(hd_dim=1000, seed=42)

# Create spectral attention with 4 heads, 3 scales
attn = SpectralAttention(
    base_graph,
    hidden_dim=64,
    num_heads=4,
    num_scales=3,
)

# Generate synthetic token embeddings (16 tokens, 64-dim)
rng = np.random.default_rng(42)
num_tokens = 16
hidden_dim = 64
tokens = rng.standard_normal((num_tokens, hidden_dim)) * 0.1
print(f"Token embeddings shape: {tokens.shape}")

# Build the graph from tokens
token_graph = attn.build_graph_from_tokens(tokens)
print(f"Built graph: {token_graph.num_nodes} nodes, {token_graph.num_edges} edges")

Token embeddings shape: (16, 64)
Built graph: 16 nodes, 63 edges


## 2. Spectral Attention Forward Pass

The forward pass performs multi-head spectral attention:

1. Project embeddings into Q, K, V spaces.
2. For each head, compute attention in the spectral domain using
   Laplacian eigenvectors.
3. Concatenate heads and project to output.

This avoids the O(N²) dot-product attention and uses graph structure
for efficient computation.

In [3]:
# Run the forward pass
output = attn.forward(tokens)
print(f"Input shape:  {tokens.shape}")
print(f"Output shape: {output.shape}")
print(f"\nFirst token input (first 5 dims):  {tokens[0, :5]}")
print(f"First token output (first 5 dims): {output[0, :5]}")

Input shape:  (16, 64)
Output shape: (16, 64)

First token input (first 5 dims):  [ 0.0305 -0.104   0.075   0.0941 -0.1951]
First token output (first 5 dims): [ 0.004   0.0078  0.0056 -0.0057  0.0009]


In [4]:
# Compare spectral attention output to standard attention (baseline)
# Standard (dense) attention for comparison
head_dim = hidden_dim // attn.num_heads

# Compute standard attention scores for head 0
Q = tokens @ attn._W_q
K = tokens @ attn._W_k
V = tokens @ attn._W_v

Q_h = Q[:, :head_dim]
K_h = K[:, :head_dim]
V_h = V[:, :head_dim]

scores = Q_h @ K_h.T / np.sqrt(head_dim)
weights_standard = attn._softmax(scores)
output_standard = weights_standard @ V_h

print(f"Standard attention weights shape: {weights_standard.shape}")
print(f"Attention distribution for token 0 (first 5):")
for i in range(min(5, num_tokens)):
    print(f"  token {i}: {weights_standard[0, i]:.4f}")

Standard attention weights shape: (16, 16)
Attention distribution for token 0 (first 5):
  token 0: 0.0625
  token 1: 0.0625
  token 2: 0.0625
  token 3: 0.0625
  token 4: 0.0625


## 3. Hierarchical Coarsening (Graph Pyramid)

Spectral attention builds a hierarchical graph pyramid via **spectral coarsening**.
Using heavy-edge matching on the Fiedler vector (second-smallest Laplacian eigenvector),
nodes are paired and merged into coarser representations.

Each level of the pyramid captures structure at a different scale.

In [5]:
# Build hierarchical coarsening pyramid
levels = attn.hierarchical_coarsening()

print("Graph Pyramid (Spectral Coarsening):")
print("=" * 50)
for i, level_graph in enumerate(levels):
    label = "Original" if i == 0 else f"Coarse level {i}"
    print(f"  {label:20s}: {level_graph.num_nodes:4d} nodes, {level_graph.num_edges:4d} edges")

Graph Pyramid (Spectral Coarsening):
  Original            :   16 nodes,   63 edges
  Coarse level 1      :    8 nodes,   18 edges
  Coarse level 2      :    4 nodes,    0 edges
  Coarse level 3      :    2 nodes,    0 edges


In [6]:
# Examine the spectral structure at each level
from smgp.core.spectral import SpectralMethods

print("\nEigenvalue statistics at each level:")
print("=" * 50)
for i, level_graph in enumerate(levels):
    if level_graph.num_nodes < 2:
        print(f"  Level {i}: too few nodes for spectral analysis")
        continue
    try:
        spectral = SpectralMethods(level_graph, num_eigenvalues=min(8, level_graph.num_nodes - 1))
        eigenvalues, eigenvectors = spectral.compute_eigen()
        print(f"  Level {i}: {len(eigenvalues)} eigenvalues")
        print(f"    Min: {eigenvalues[0]:.4f}, Max: {eigenvalues[-1]:.4f}")
        if len(eigenvalues) > 1:
            print(f"    Spectral gap (lambda_2 - lambda_1): {eigenvalues[1] - eigenvalues[0]:.4f}")
    except ValueError as e:
        print(f"  Level {i}: Error — {e}")


Eigenvalue statistics at each level:
  Level 0: 8 eigenvalues
    Min: -0.0000, Max: 1.1327
    Spectral gap (lambda_2 - lambda_1): 0.2948
  Level 1: 7 eigenvalues
    Min: -0.0000, Max: 1.4129
    Spectral gap (lambda_2 - lambda_1): 0.5696
  Level 2: 3 eigenvalues
    Min: 1.0000, Max: 1.0000
    Spectral gap (lambda_2 - lambda_1): 0.0000
  Level 3: 1 eigenvalues
    Min: 1.0000, Max: 1.0000


## 4. Comparing Attention Weights Across Scales

Spectral attention produces different attention patterns at different
scales of the graph pyramid. Lower levels capture local structure
while higher levels capture global relationships.

In [7]:
# Run spectral attention at each pyramid level
print("Attention output norms at each pyramid level:")
print("=" * 50)

for i, level_graph in enumerate(levels):
    if level_graph.num_nodes < 2:
        continue
    level_attn = SpectralAttention(
        level_graph,
        hidden_dim=64,
        num_heads=4,
        num_scales=2,
    )
    # Create random embeddings for this level
    n = level_graph.num_nodes
    level_embeddings = rng.standard_normal((n, 64)) * 0.1
    level_output = level_attn.forward(level_embeddings)
    
    label = "Original" if i == 0 else f"Coarse level {i}"
    norms = np.linalg.norm(level_output, axis=1)
    print(f"  {label:20s}: mean_norm={norms.mean():.4f}, std_norm={norms.std():.4f}")

Attention output norms at each pyramid level:
  Original            : mean_norm=0.1259, std_norm=0.0182
  Coarse level 1      : mean_norm=0.1261, std_norm=0.0126
  Coarse level 2      : mean_norm=0.1244, std_norm=0.0047
  Coarse level 3      : mean_norm=0.1377, std_norm=0.0173


In [8]:
# Visualize the token graph adjacency structure
print("Token graph edge list (sample):")
edges = list(token_graph.edges())[:10]  # Show first 10 edges
for src, tgt, key, data in edges:
    relation = data.get("relation", "?")
    print(f"  {src} --[{relation}]--> {tgt}")
print(f"  ... ({token_graph.num_edges} total edges)")

Token graph edge list (sample):
  token_0 --[similar]--> token_2
  token_0 --[similar]--> token_15
  token_0 --[similar]--> token_9
  token_0 --[sequential]--> token_1
  token_1 --[similar]--> token_9
  token_1 --[similar]--> token_2
  token_1 --[sequential]--> token_2
  token_1 --[similar]--> token_4
  token_2 --[similar]--> token_0
  token_2 --[similar]--> token_14
  ... (63 total edges)


In [9]:
# Demonstrate graph wavelet transform
spectral = SpectralMethods(token_graph, num_eigenvalues=min(8, num_tokens - 1))
signal = rng.standard_normal(num_tokens)

wavelet_coeffs = spectral.graph_wavelet_transform(signal, scales=[1.0, 2.0, 4.0])
print(f"Wavelet coefficients shape: {wavelet_coeffs.shape}")
print(f"  (num_scales={wavelet_coeffs.shape[0]}, num_nodes={wavelet_coeffs.shape[1]})")
print(f"\nCoefficient norms per scale:")
for i, s in enumerate([1.0, 2.0, 4.0]):
    print(f"  Scale {s:.1f}: mean={np.abs(wavelet_coeffs[i]).mean():.4f}, "
          f"max={np.abs(wavelet_coeffs[i]).max():.4f}")

Wavelet coefficients shape: (3, 16)
  (num_scales=3, num_nodes=16)

Coefficient norms per scale:
  Scale 1.0: mean=0.3356, max=0.7943
  Scale 2.0: mean=0.1891, max=0.3466
  Scale 4.0: mean=0.0818, max=0.1431


## Summary

This notebook demonstrated:

1. **Graph construction from tokens**: Building a context graph with k-NN edges.
2. **Spectral attention forward pass**: Multi-head attention via Laplacian eigenvectors.
3. **Hierarchical coarsening**: Building a graph pyramid via Fiedler-vector matching.
4. **Multi-scale analysis**: Comparing attention patterns and wavelet coefficients
   across different pyramid levels.

The spectral approach provides O(N log N) complexity by exploiting graph structure,
making it suitable for long-context reasoning tasks.